# Semana 07: Publicação de Imagens: Docker Hub e GitHub Actions

## Módulo de Registro e Publicação de Containers — Fábrica Virtual Smart N1

Este notebook apresenta a automação de **Build e Push de imagens de container para o Docker Hub** via **GitHub Actions**, abordando autenticação por tokens seguros, versionamento semântico de tags (`v1.0.0`, `latest`) e a segurança no armazenamento de credenciais no repositório.

### Objetivos de aprendizagem
- Compreender o papel dos registros de imagens de container (*Container Registries* - Docker Hub, GHCR, ECR).
- Configurar autenticação segura no Docker Hub via GitHub Actions Secrets (`DOCKER_USERNAME` e `DOCKER_PASSWORD` / Access Token).
- Aplicar estratégias de versionamento de imagens com tags (`v1.0.0`, `sha-git`, `latest`).
- Utilizar as Actions oficiais da Docker (`docker/login-action`, `docker/build-push-action`).
- Executar um simulador em Python que valida e gera os comandos de publicação no Docker Hub.

---


## 1. Fundamentação Teórica

### 1.1 O Papel do Container Registry no Pipeline de CI/CD

Um **Container Registry** (como o **Docker Hub**) serve como o repositório centralizado de imagens prontas para execução em ambientes de staging e produção:

```text
  +-------------------------------------------------------------------------+
  |                PIPELINE DE BUILD E PUSH DOCKER HUB                      |
  |                                                                         |
  |  [Git Push Tag: v1.0.0] ---> [GitHub Actions Runner]                    |
  |                                       |                                 |
  |                                       v                                 |
  |                       1. docker/login-action                            |
  |                          (Autentica via Secrets)                        |
  |                                       |                                 |
  |                                       v                                 |
  |                       2. docker/build-push-action                       |
  |                          (Compila imagem Multi-Stage)                   |
  |                                       |                                 |
  |                                       v                                 |
  |                       3. Push Tags: [v1.0.0, latest]                    |
  |                                       |                                 |
  |                                       v                                 |
  |                             [DOCKER HUB REGISTRY]                       |
  |                    (usuario/smartn1-telemetria:v1.0.0)                  |
  +-------------------------------------------------------------------------+
```

---

### 1.2 Exemplo de Workflow YAML para Publicação no Docker Hub

```yaml
name: Publish Container Image to Docker Hub

on:
  push:
    tags:
      - 'v*.*.*'

jobs:
  build-and-push:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout Code
        uses: actions/checkout@v4

      - name: Login to Docker Hub
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKERHUB_USERNAME }}
          password: ${{ secrets.DOCKERHUB_TOKEN }}

      - name: Extract Metadata (Tags & Labels)
        id: meta
        uses: docker/metadata-action@v5
        with:
          images: usuario/smartn1-telemetria

      - name: Build and Push Docker Image
        uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          tags: ${{ steps.meta.outputs.tags }}
          labels: ${{ steps.meta.outputs.labels }}
```

---


## 2. Prática — Gerador e Validador de Publicação Docker Hub em Python

Nesta prática, escreveremos um script Python que simula a etiquetagem de imagens Docker (*tags*) a partir de eventos do Git e valida os comandos de `docker build`, `docker tag` e `docker push`.

In [ ]:
def simular_push_docker_hub(usuario_docker, nome_imagem, git_tag, commit_sha):
    # Gerar a lista de tags oficiais a serem publicadas
    tags = [
        f"{usuario_docker}/{nome_imagem}:{git_tag}",
        f"{usuario_docker}/{nome_imagem}:sha-{commit_sha[:7]}",
        f"{usuario_docker}/{nome_imagem}:latest"
    ]
    
    comandos_executados = []
    comandos_executados.append(f"docker build -t {tags[0]} .")
    for t in tags[1:]:
        comandos_executados.append(f"docker tag {tags[0]} {t}")
    for t in tags:
        comandos_executados.append(f"docker push {t}")
        
    return {
        "tags_geradas": tags,
        "total_comandos": len(comandos_executados),
        "comandos": comandos_executados
    }

resultado_pub = simular_push_docker_hub("smartn1oficial", "telemetria-api", "v1.2.0", "9f8e7d6c5b4a")

print("=== SIMULAÇÃO DE PUBLICADOR AUTOMÁTICO DOCKER HUB ===\n")
print("Tags que serão publicadas no Docker Hub:")
for tag in resultado_pub["tags_geradas"]:
    print(f"  - {tag}")
    
print("\nComandos executados pelo Runner do GitHub Actions:")
for cmd in resultado_pub["comandos"]:
    print(f"  $ {cmd}")


---

## 3. Exercícios de Fixação e Avaliação

### Questão 1
Por que é uma boa prática publicar a mesma imagem de container com múltiplas tags (ex: `v1.2.0`, `sha-9f8e7d6` e `latest`) no Docker Hub?

### Questão 2
Qual a diferença de segurança entre autenticar no Docker Hub via senha pessoal do usuário e autenticar utilizando um **Access Token (Personal Access Token - PAT)** gerado com permissões restritas de *Write*?

### Questão 3
Como configurar o workflow do GitHub Actions para que a etapa de `docker push` seja executada **apenas** quando uma tag de versão (ex: `v1.0.0`) for publicada na branch `main`?
